# 10 - Attention

**AI sin humo** - Notas personales para entender deep learning desde cero.

En el notebook anterior vimos que las RNNs y LSTMs tienen un problema fundamental: **toda la información de la secuencia se comprime en un solo vector** (el hidden state final). Esto es un bottleneck brutal. Si estás traduciendo un párrafo de 200 palabras, toda la información tiene que caber en un vector de dimensión 512. Es como resumir un libro en un tweet.

Attention es la solución elegante a este problema. La idea es simple pero revolucionaria: **¿por qué comprimir todo si el decoder puede mirar directamente cualquier parte del input cuando lo necesite?** En vez de un resumen fijo, le damos al decoder acceso directo a cada token del encoder, y dejamos que aprenda a qué partes prestar atención en cada paso.

Y lo que empezó como un parche para mejorar Seq2Seq terminó siendo **la idea más importante del deep learning moderno**. Attention no solo mejoró las RNNs — las reemplazó por completo. Los Transformers (GPT, BERT, etc.) son, en esencia, attention pura, sin recurrencia.

---

## Contenido

1. [El problema que resuelve attention](#problema)
2. [Bahdanau attention (2014)](#bahdanau)
3. [Dot product attention (Luong)](#luong)
4. [Self-attention](#self-attention)
5. [Query, Key, Value con transformaciones aprendibles](#qkv)
6. [¿Por qué Q, K, V distintos?](#por-que-qkv)
7. [Múltiples capas de attention](#capas)
8. [Conclusiones](#conclusiones)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

torch.manual_seed(42)
np.random.seed(42)

---

<a id='problema'></a>
## 1. El problema que resuelve attention

### El hidden state bottleneck

Recapitulemos rápido dónde estamos. En el notebook anterior terminamos con Seq2Seq: un encoder RNN/LSTM que procesa la secuencia de entrada y produce un hidden state final, y un decoder RNN/LSTM que recibe ese hidden state y genera la secuencia de salida.

El problema fatal:

$$\underbrace{\mathbf{x}_1, \mathbf{x}_2, \mathbf{x}_3, \ldots, \mathbf{x}_T}_{\text{secuencia completa}} \xrightarrow{\text{Encoder RNN}} \underbrace{\mathbf{h}_T}_{\text{un solo vector}} \xrightarrow{\text{Decoder RNN}} \underbrace{\mathbf{y}_1, \mathbf{y}_2, \ldots, \mathbf{y}_{T'}}_{\text{secuencia de salida}}$$

**Toda** la información de la secuencia de entrada tiene que pasar por un solo vector $\mathbf{h}_T$ de dimensión fija. Da igual si la oración tiene 5 palabras o 500 — el cuello de botella es el mismo vector.

### ¿Por qué es tan grave?

Pensá en traducción. Estás traduciendo del español al inglés:

> "El gato negro que estaba sentado en el tejado de la casa vieja se bajó corriendo cuando empezó la tormenta"

El encoder procesa toda esta oración y la comprime en un vector de, digamos, 512 dimensiones. Después, el decoder tiene que generar la traducción **usando solo ese vector**. 

Cuando el decoder está generando la palabra "cat", necesita información sobre "gato". Cuando está generando "roof", necesita "tejado". Cuando genera "storm", necesita "tormenta". Pero **toda esa información está mezclada y comprimida en un solo vector**. El decoder no puede decir "che, necesito más detalle sobre la parte que hablaba del tejado".

Es como si un intérprete escuchara todo tu discurso de una hora, lo resumiera mentalmente en un párrafo, y después tratara de traducirlo. Obviamente va a perder detalles.

### La pregunta que cambió todo

La pregunta obvia pero revolucionaria que se hicieron Bahdanau, Cho y Bengio en 2014 fue:

> **¿Por qué no permitir que el decoder mire directamente cada hidden state del encoder?**

El encoder genera un hidden state $\mathbf{h}_t^{enc}$ en cada paso temporal. Esos hidden states contienen información rica sobre cada parte del input. ¿Por qué tirar todo eso y quedarnos solo con el último?

```
ANTES (Seq2Seq vanilla):
  Encoder: h1 → h2 → h3 → h4 → h5 ──▶ [solo h5 pasa al decoder]
                                              │
  Decoder:                                    ▼
                                         genera output

DESPUÉS (con Attention):
  Encoder: h1   h2   h3   h4   h5  ← [TODOS disponibles]
            \    \    |    /    /
             \    \   |   /   /
              ▼    ▼  ▼  ▼   ▼       ← en cada paso, el decoder
  Decoder:   [mira todos y elige]       decide a cuáles prestar
                     │                  atención
                     ▼
                genera output
```

En vez de comprimir todo en un solo vector, le damos al decoder acceso directo a **toda la memoria del encoder**. Y en cada paso de decodificación, el decoder decide dinámicamente a qué partes del input prestar atención.

Eso es attention: una **vista enfocada** sobre el input que cambia en cada paso de generación.

In [ ]:
# Demo: the bottleneck problem — visualizing information loss
# How much of the original sequence can a single hidden state preserve?

def measure_bottleneck(seq_lengths, hidden_dim=64, embed_dim=32, vocab_size=100, n_trials=5):
    """Measure how well a single hidden state preserves sequence information."""
    results = {}
    
    for seq_len in seq_lengths:
        accs = []
        for trial in range(n_trials):
            torch.manual_seed(trial)
            
            # Encoder LSTM
            embedding = nn.Embedding(vocab_size, embed_dim)
            encoder = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
            # Try to reconstruct which tokens appeared at which positions
            decoder_fc = nn.Linear(hidden_dim, vocab_size * seq_len)
            
            params = list(embedding.parameters()) + list(encoder.parameters()) + list(decoder_fc.parameters())
            optimizer = torch.optim.Adam(params, lr=0.005)
            
            for step in range(200):
                x = torch.randint(0, vocab_size, (32, seq_len))
                emb = embedding(x)
                _, (h, _) = encoder(emb)
                # Try to predict all tokens from just the final hidden state
                logits = decoder_fc(h.squeeze(0))  # (batch, vocab_size * seq_len)
                logits = logits.view(-1, seq_len, vocab_size)  # (batch, seq_len, vocab_size)
                
                loss = F.cross_entropy(logits.reshape(-1, vocab_size), x.reshape(-1))
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            
            # Evaluate
            with torch.no_grad():
                x = torch.randint(0, vocab_size, (200, seq_len))
                emb = embedding(x)
                _, (h, _) = encoder(emb)
                logits = decoder_fc(h.squeeze(0)).view(-1, seq_len, vocab_size)
                preds = logits.argmax(dim=-1)
                acc = (preds == x).float().mean().item()
                accs.append(acc)
        
        results[seq_len] = np.mean(accs)
    
    return results

seq_lengths = [3, 5, 10, 20, 40]
results = measure_bottleneck(seq_lengths)

fig, ax = plt.subplots(figsize=(9, 5))
lengths = list(results.keys())
accs = [results[l] for l in lengths]

colors = ['#27ae60' if a > 0.5 else '#e67e22' if a > 0.2 else '#e74c3c' for a in accs]
bars = ax.bar(range(len(lengths)), accs, color=colors, edgecolor='white', linewidth=1.5)
ax.set_xticks(range(len(lengths)))
ax.set_xticklabels([str(l) for l in lengths])
ax.set_xlabel('Sequence length', fontsize=12)
ax.set_ylabel('Reconstruction accuracy', fontsize=12)
ax.set_title('The Hidden State Bottleneck\nCan a single vector remember the whole sequence?', fontsize=13)
ax.set_ylim(0, 1.05)

for i, acc in enumerate(accs):
    ax.text(i, acc + 0.02, f'{acc:.1%}', ha='center', fontsize=11, fontweight='bold')

# Add annotation
ax.annotate('Longer sequences → more info lost!', 
            xy=(3, accs[3]), xytext=(1.5, 0.85),
            arrowprops=dict(arrowstyle='->', color='red', lw=1.5),
            fontsize=11, color='red')

plt.tight_layout()
plt.show()

print("The longer the sequence, the less a single hidden state can remember.")
print("This is why we need attention: direct access to all encoder states.")

---

<a id='bahdanau'></a>
## 2. Bahdanau Attention (2014)

### La primera solución al bottleneck

Bahdanau, Cho y Bengio publicaron "Neural Machine Translation by Jointly Learning to Align and Translate" en 2014, y con ese paper cambiaron todo. La idea central es:

> **En cada paso del decoder, calcular un "context vector" que es un promedio ponderado de TODOS los hidden states del encoder.** Los pesos de ese promedio se calculan dinámicamente — el modelo aprende a qué parte del input prestar atención.

### El mecanismo paso a paso

Supongamos que el encoder procesó una secuencia de $T$ tokens y produjo hidden states $\mathbf{h}_1^{enc}, \mathbf{h}_2^{enc}, \ldots, \mathbf{h}_T^{enc}$.

En cada paso $t$ del decoder (cuando estamos generando el token $y_t$), hacemos:

**Paso 1: Calcular scores de afinidad ("¿cuánto me interesa cada parte del input?")**

Para cada hidden state del encoder $\mathbf{h}_j^{enc}$, calculamos un score que indica qué tan relevante es para el paso actual del decoder:

$$e_{t,j} = \mathbf{v}^T \tanh(\mathbf{W}_1 \cdot \mathbf{s}_{t-1} + \mathbf{W}_2 \cdot \mathbf{h}_j^{enc})$$

Donde:
- $\mathbf{s}_{t-1}$ es el hidden state actual del decoder (lo que el decoder "está pensando")
- $\mathbf{h}_j^{enc}$ es el hidden state del encoder en la posición $j$ (información sobre esa parte del input)
- $\mathbf{W}_1$, $\mathbf{W}_2$, $\mathbf{v}$ son parámetros aprendibles (una mini red feedforward)

Fijate que esto es una **red neuronal pequeña** que recibe dos vectores y produce un escalar. Aprende a medir "¿qué tan útil es $\mathbf{h}_j^{enc}$ dado que el decoder está en el estado $\mathbf{s}_{t-1}$?".

**Paso 2: Normalizar con softmax → pesos de atención**

Los scores $e_{t,j}$ se normalizan para que sumen 1:

$$\alpha_{t,j} = \frac{\exp(e_{t,j})}{\sum_{k=1}^{T} \exp(e_{t,k})} = \text{softmax}(e_{t,:})_j$$

Los $\alpha_{t,j}$ son los **pesos de atención**: una distribución de probabilidad sobre las posiciones del input. Si $\alpha_{t,3}$ es alto, significa que el decoder está prestando mucha atención al token 3 del input en este paso.

**Paso 3: Context vector = promedio ponderado**

$$\mathbf{c}_t = \sum_{j=1}^{T} \alpha_{t,j} \cdot \mathbf{h}_j^{enc}$$

El context vector $\mathbf{c}_t$ es un promedio ponderado de todos los hidden states del encoder, donde los pesos son los pesos de atención. Es una **vista enfocada** de la secuencia de entrada, personalizada para lo que el decoder necesita en este momento.

**Paso 4: Usar el context vector en el decoder**

El context vector se concatena con el input del decoder y se usa para generar el siguiente token:

$$\mathbf{s}_t = \text{RNN}([y_{t-1}; \mathbf{c}_t], \mathbf{s}_{t-1})$$

### Intuición visual

```
Encoder hidden states:   h1    h2    h3    h4    h5
                          │     │     │     │     │
Attention weights:      0.05  0.10  0.60  0.20  0.05  ← softmax (suman 1)
                          │     │     │     │     │
                          ▼     ▼     ▼     ▼     ▼
Context vector c_t = 0.05·h1 + 0.10·h2 + 0.60·h3 + 0.20·h4 + 0.05·h5
                                          ↑
                                    El decoder está enfocado
                                    principalmente en h3
```

Cada paso del decoder produce sus propios pesos de atención. Cuando el decoder está traduciendo "cat", pone peso alto en el token "gato". Cuando traduce "roof", pone peso alto en "tejado". La atención se **mueve** a lo largo del input según lo que el decoder necesita.

In [ ]:
# Bahdanau Attention — full implementation from scratch

class BahdanauAttention(nn.Module):
    """Additive attention (Bahdanau et al., 2014).
    
    Score function: e_{t,j} = v^T · tanh(W1 · s_{t-1} + W2 · h_j^enc)
    This is a small neural network that learns to score relevance.
    """
    def __init__(self, encoder_dim, decoder_dim, attention_dim):
        super().__init__()
        self.W1 = nn.Linear(decoder_dim, attention_dim, bias=False)  # transform decoder state
        self.W2 = nn.Linear(encoder_dim, attention_dim, bias=False)  # transform encoder states
        self.v = nn.Linear(attention_dim, 1, bias=False)             # produce scalar score
    
    def forward(self, decoder_state, encoder_outputs):
        """
        Args:
            decoder_state: (batch, decoder_dim) — current decoder hidden state
            encoder_outputs: (batch, src_len, encoder_dim) — all encoder hidden states
        Returns:
            context: (batch, encoder_dim) — weighted sum of encoder outputs
            weights: (batch, src_len) — attention weights (sum to 1)
        """
        # decoder_state: (batch, decoder_dim) → (batch, 1, attention_dim)
        query = self.W1(decoder_state).unsqueeze(1)
        
        # encoder_outputs: (batch, src_len, encoder_dim) → (batch, src_len, attention_dim)
        keys = self.W2(encoder_outputs)
        
        # Score: v^T · tanh(query + keys)
        # query broadcasts: (batch, 1, attn_dim) + (batch, src_len, attn_dim)
        scores = self.v(torch.tanh(query + keys))  # (batch, src_len, 1)
        scores = scores.squeeze(-1)                 # (batch, src_len)
        
        # Softmax → attention weights
        weights = F.softmax(scores, dim=-1)  # (batch, src_len)
        
        # Context vector = weighted sum of encoder outputs
        # weights: (batch, src_len) → (batch, 1, src_len)
        # encoder_outputs: (batch, src_len, encoder_dim)
        context = torch.bmm(weights.unsqueeze(1), encoder_outputs)  # (batch, 1, encoder_dim)
        context = context.squeeze(1)  # (batch, encoder_dim)
        
        return context, weights


# Demo: step-by-step Bahdanau attention
torch.manual_seed(42)

batch_size = 1
src_len = 6
encoder_dim = 8
decoder_dim = 8
attention_dim = 4

# Simulate encoder outputs (as if an RNN processed 6 tokens)
encoder_outputs = torch.randn(batch_size, src_len, encoder_dim)

# Simulate decoder state (current state of the decoder)
decoder_state = torch.randn(batch_size, decoder_dim)

# Create attention module
attention = BahdanauAttention(encoder_dim, decoder_dim, attention_dim)

# Forward pass
context, weights = attention(decoder_state, encoder_outputs)

print("Bahdanau Attention — Step by Step")
print("=" * 50)
print(f"\nEncoder outputs shape: {encoder_outputs.shape}  (batch, src_len={src_len}, enc_dim={encoder_dim})")
print(f"Decoder state shape:   {decoder_state.shape}  (batch, dec_dim={decoder_dim})")
print(f"\nAttention weights: {weights.detach().numpy().round(3)}")
print(f"  Sum = {weights.sum().item():.4f}  (should be 1.0)")
print(f"  Most attended position: {weights.argmax().item()}")
print(f"\nContext vector shape: {context.shape}  (batch, enc_dim={encoder_dim})")
print(f"Context vector: {context.detach().numpy().round(3)}")

In [ ]:
# Visualize: how attention shifts as the decoder generates each output token
# We simulate a simple translation scenario

torch.manual_seed(42)

# Source sentence tokens
src_tokens = ["El", "gato", "negro", "duerme", "tranquilo", "."]
src_len = len(src_tokens)

# Target tokens (what the decoder generates)
tgt_tokens = ["The", "black", "cat", "sleeps", "quietly", "."]

# Simulate encoder outputs and decoder states at each step
encoder_dim = 16
decoder_dim = 16
attention_dim = 8

encoder_outputs = torch.randn(1, src_len, encoder_dim)
attention_module = BahdanauAttention(encoder_dim, decoder_dim, attention_dim)

# Collect attention weights for each decoder step
all_weights = []
for t in range(len(tgt_tokens)):
    # Each decoder step has a different hidden state
    torch.manual_seed(t * 10 + 7)  # different state per step
    decoder_state = torch.randn(1, decoder_dim)
    _, weights = attention_module(decoder_state, encoder_outputs)
    all_weights.append(weights.detach().numpy().flatten())

attention_matrix = np.array(all_weights)

# Plot attention heatmap
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(attention_matrix, cmap='Blues', aspect='auto')
ax.set_xticks(range(src_len))
ax.set_xticklabels(src_tokens, fontsize=12)
ax.set_yticks(range(len(tgt_tokens)))
ax.set_yticklabels(tgt_tokens, fontsize=12)
ax.set_xlabel('Source (Encoder)', fontsize=13)
ax.set_ylabel('Target (Decoder)', fontsize=13)
ax.set_title('Bahdanau Attention Weights\n(which source tokens does the decoder look at?)', fontsize=14)

# Add text annotations
for i in range(len(tgt_tokens)):
    for j in range(src_len):
        val = attention_matrix[i, j]
        color = 'white' if val > 0.3 else 'black'
        ax.text(j, i, f'{val:.2f}', ha='center', va='center', fontsize=10, color=color)

plt.colorbar(im, ax=ax, label='Attention weight')
plt.tight_layout()
plt.show()

print("Each row shows where the decoder 'looks' when generating that output token.")
print("In a trained model, you'd see diagonal-ish patterns for translation,")
print("with the decoder focusing on the corresponding source word.")

In [ ]:
# Full Seq2Seq with Bahdanau Attention

class AttentionEncoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True, bidirectional=False)
    
    def forward(self, x):
        embedded = self.embedding(x)  # (batch, src_len, embed_dim)
        outputs, (hidden, cell) = self.lstm(embedded)
        # outputs: (batch, src_len, hidden_dim) — ALL hidden states
        # hidden: (1, batch, hidden_dim) — last hidden state only
        return outputs, hidden, cell


class AttentionDecoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, attention_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.attention = BahdanauAttention(hidden_dim, hidden_dim, attention_dim)
        # Input to LSTM: embedding + context vector
        self.lstm = nn.LSTM(embed_dim + hidden_dim, hidden_dim, batch_first=True)
        self.fc_out = nn.Linear(hidden_dim, vocab_size)
    
    def forward_step(self, input_token, hidden, cell, encoder_outputs):
        """One decoding step with attention."""
        # Embed the input token
        embedded = self.embedding(input_token)  # (batch, 1, embed_dim)
        
        # Compute attention using current decoder hidden state
        context, attn_weights = self.attention(
            hidden.squeeze(0),   # (batch, hidden_dim)
            encoder_outputs       # (batch, src_len, hidden_dim)
        )
        
        # Concatenate embedding and context, then feed to LSTM
        lstm_input = torch.cat([embedded, context.unsqueeze(1)], dim=-1)  # (batch, 1, embed+hidden)
        output, (hidden, cell) = self.lstm(lstm_input, (hidden, cell))
        
        # Project to vocabulary
        logits = self.fc_out(output.squeeze(1))  # (batch, vocab_size)
        
        return logits, hidden, cell, attn_weights


# Create model
vocab_size = 100
embed_dim = 32
hidden_dim = 64
attention_dim = 32

enc = AttentionEncoder(vocab_size, embed_dim, hidden_dim)
dec = AttentionDecoder(vocab_size, embed_dim, hidden_dim, attention_dim)

# Demo forward pass
src = torch.randint(0, vocab_size, (1, 8))  # source: 8 tokens
trg = torch.randint(0, vocab_size, (1, 6))  # target: 6 tokens

# Encode
enc_outputs, hidden, cell = enc(src)
print(f"Encoder outputs: {enc_outputs.shape}  ← ALL hidden states kept!")
print(f"Encoder final hidden: {hidden.shape}")

# Decode step by step (with attention)
print(f"\nDecoding with attention, step by step:")
input_token = trg[:, 0:1]  # start with first target token

for t in range(1, trg.shape[1]):
    logits, hidden, cell, attn_w = dec.forward_step(input_token, hidden, cell, enc_outputs)
    input_token = trg[:, t:t+1]  # teacher forcing
    
    top_token = logits.argmax(dim=-1).item()
    top_attn = attn_w.argmax(dim=-1).item()
    print(f"  Step {t}: predicted token={top_token}, most attended source position={top_attn}")
    print(f"           attention weights: {attn_w.detach().numpy().round(3)}")

print(f"\n✓ Unlike vanilla Seq2Seq, the decoder now has access to ALL encoder states!")
print(f"  No more bottleneck — each step looks at the full input.")

### ¿Cómo se aprenden los pesos de atención?

Esto es lo elegante: **los pesos de atención se aprenden end-to-end con backpropagation**. No hay supervisión explícita que diga "en este paso, mirá el token 3". El modelo aprende solito a dónde mirar, porque los pesos de atención ($\mathbf{W}_1, \mathbf{W}_2, \mathbf{v}$) se actualizan durante el entrenamiento para minimizar el loss de traducción.

Si el modelo presta atención al token equivocado → genera una palabra incorrecta → loss alto → gradiente grande → actualiza los pesos de atención para que la próxima vez mire al token correcto.

Es un mecanismo de **alineamiento aprendido**: el modelo descubre la correspondencia entre tokens del input y del output sin que nadie se la enseñe explícitamente.

### El resultado: mejora dramática

El paper de Bahdanau mostró mejoras significativas en traducción automática, especialmente para **oraciones largas**. Mientras que el Seq2Seq vanilla se degradaba rápidamente después de ~20 tokens, el modelo con attention mantenía calidad incluso en oraciones de 50+ tokens.

¿Por qué? Porque ya no depende de comprimir todo en un vector. Para cada token de salida, tiene acceso directo a la información relevante del input.

---

<a id='luong'></a>
## 3. Dot Product Attention (Luong)

### Simplificando el score

Bahdanau attention usa una red feedforward para calcular los scores:

$$e_{t,j} = \mathbf{v}^T \tanh(\mathbf{W}_1 \cdot \mathbf{s}_{t-1} + \mathbf{W}_2 \cdot \mathbf{h}_j^{enc})$$

Esto funciona bien, pero involucra parámetros extra ($\mathbf{W}_1, \mathbf{W}_2, \mathbf{v}$) y operaciones no lineales (tanh). Luong et al. (2015) propusieron algo más simple: **¿y si el score es simplemente el dot product entre los dos vectores?**

$$e_{t,j} = \mathbf{s}_{t-1}^T \cdot \mathbf{h}_j^{enc}$$

Eso es todo. El dot product ya mide similitud entre vectores: si apuntan en la misma dirección, el score es alto. Si son ortogonales, el score es cero.

### Query, Key, Value — los nombres que cambiaron todo

Luong introdujo una terminología que se volvió estándar:

| Concepto | En Luong attention | Descripción |
|:---------|:-------------------|:------------|
| **Query** (Q) | $\mathbf{s}_{t-1}$ (decoder state) | Lo que estoy buscando |
| **Key** (K) | $\mathbf{h}_j^{enc}$ (encoder states) | Contra qué comparo |
| **Value** (V) | $\mathbf{h}_j^{enc}$ (encoder states) | La información que extraigo |

Fijate algo importante: en esta versión, **Keys y Values son lo mismo** (ambos son los hidden states del encoder). Esto cambiará después.

### El mecanismo

$$\text{score}(Q, K_j) = Q^T \cdot K_j \quad \text{(dot product)}$$
$$\alpha_j = \text{softmax}(\text{scores})_j$$
$$\text{context} = \sum_j \alpha_j \cdot V_j$$

Es la misma estructura que Bahdanau (score → softmax → weighted sum), pero con un score mucho más simple.

### Comparación Bahdanau vs Luong

| Aspecto | Bahdanau (additive) | Luong (dot product) |
|:--------|:-------------------|:--------------------|
| **Score function** | $\mathbf{v}^T \tanh(\mathbf{W}_1 s + \mathbf{W}_2 h)$ | $s^T \cdot h$ |
| **Parámetros extra** | $\mathbf{W}_1, \mathbf{W}_2, \mathbf{v}$ | Ninguno |
| **Velocidad** | Más lento (red feedforward) | Más rápido (solo dot product) |
| **Expresividad** | Más expresivo (función no lineal) | Menos expresivo, pero suficiente |
| **En la práctica** | Funciona bien, históricamente primero | Base de la attention moderna |

In [ ]:
# Luong (Dot Product) Attention — implementation and comparison

class LuongDotAttention(nn.Module):
    """Dot product attention (Luong et al., 2015).
    
    Score function: e_{t,j} = s_{t-1}^T · h_j^enc
    No extra parameters needed! Just a dot product.
    """
    def forward(self, query, keys, values=None):
        """
        Args:
            query: (batch, dim) — decoder hidden state
            keys: (batch, src_len, dim) — encoder hidden states
            values: (batch, src_len, dim) — same as keys in Luong
        """
        if values is None:
            values = keys  # In Luong, K == V
        
        # Dot product: query @ keys^T
        # query: (batch, 1, dim) @ keys^T: (batch, dim, src_len) = (batch, 1, src_len)
        scores = torch.bmm(query.unsqueeze(1), keys.transpose(1, 2))  # (batch, 1, src_len)
        scores = scores.squeeze(1)  # (batch, src_len)
        
        # Softmax
        weights = F.softmax(scores, dim=-1)
        
        # Weighted sum
        context = torch.bmm(weights.unsqueeze(1), values).squeeze(1)  # (batch, dim)
        
        return context, weights


# Compare Bahdanau vs Luong
torch.manual_seed(42)

batch_size = 1
src_len = 8
dim = 16

query = torch.randn(batch_size, dim)
keys = torch.randn(batch_size, src_len, dim)

# Bahdanau
bahdanau = BahdanauAttention(dim, dim, dim // 2)
ctx_b, w_b = bahdanau(query, keys)

# Luong
luong = LuongDotAttention()
ctx_l, w_l = luong(query, keys)

# Compare
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ax, weights, name in zip(axes, [w_b, w_l], ['Bahdanau (additive)', 'Luong (dot product)']):
    w = weights.detach().numpy().flatten()
    bars = ax.bar(range(src_len), w, color='steelblue', edgecolor='white')
    ax.set_xlabel('Source position', fontsize=11)
    ax.set_ylabel('Attention weight', fontsize=11)
    ax.set_title(name, fontsize=13)
    ax.set_ylim(0, max(w) * 1.3)
    for i, v in enumerate(w):
        ax.text(i, v + 0.005, f'{v:.3f}', ha='center', fontsize=9)

plt.suptitle('Same query, same keys — different attention mechanisms', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

# Count parameters
bahdanau_params = sum(p.numel() for p in bahdanau.parameters())
print(f"Bahdanau parameters: {bahdanau_params}")
print(f"Luong parameters:    0  (no extra parameters!)")
print(f"\nBoth produce valid attention weights (sum to 1), but Luong is simpler and faster.")

### ¿Dot product como medida de similitud?

¿Por qué funciona usar el dot product como score? Porque el dot product mide qué tan "alineados" están dos vectores:

$$\mathbf{a}^T \cdot \mathbf{b} = \|\mathbf{a}\| \|\mathbf{b}\| \cos(\theta)$$

- Si apuntan en la **misma dirección** ($\theta \approx 0$): dot product alto y positivo → "son similares"
- Si son **ortogonales** ($\theta = 90°$): dot product ≈ 0 → "no tienen relación"
- Si apuntan en **direcciones opuestas** ($\theta \approx 180°$): dot product negativo → "son opuestos"

Cuando el decoder busca información sobre "the cat", su hidden state apunta en una dirección. Los hidden states del encoder que codifican "el gato" apuntan en una dirección similar (porque representan el mismo concepto). El dot product entre ellos es alto → peso de atención alto.

Esto es más eficiente que la red feedforward de Bahdanau, y en la práctica funciona igual de bien (o mejor, porque es más fácil de optimizar).

In [ ]:
# Geometric intuition: dot product as similarity

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

scenarios = [
    ("Same direction\n(high similarity)", [1.0, 0.5], [0.8, 0.4]),
    ("Orthogonal\n(no similarity)", [1.0, 0.0], [0.0, 1.0]),
    ("Opposite direction\n(negative similarity)", [1.0, 0.5], [-0.8, -0.4]),
]

for ax, (title, a, b) in zip(axes, scenarios):
    a, b = np.array(a), np.array(b)
    dot = np.dot(a, b)
    
    ax.arrow(0, 0, a[0], a[1], head_width=0.05, head_length=0.03, fc='#3498db', ec='#3498db', linewidth=2)
    ax.arrow(0, 0, b[0], b[1], head_width=0.05, head_length=0.03, fc='#e74c3c', ec='#e74c3c', linewidth=2)
    
    ax.text(a[0] * 0.5 + 0.05, a[1] * 0.5 + 0.1, 'query', color='#3498db', fontsize=12, fontweight='bold')
    ax.text(b[0] * 0.5 + 0.05, b[1] * 0.5 - 0.15, 'key', color='#e74c3c', fontsize=12, fontweight='bold')
    
    ax.set_xlim(-1.3, 1.3)
    ax.set_ylim(-0.8, 0.8)
    ax.set_aspect('equal')
    ax.axhline(y=0, color='gray', linestyle='-', alpha=0.3)
    ax.axvline(x=0, color='gray', linestyle='-', alpha=0.3)
    ax.set_title(f'{title}\ndot product = {dot:.2f}', fontsize=12)
    ax.grid(True, alpha=0.2)

plt.suptitle('Dot Product as Similarity Measure', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("High dot product → vectors point the same way → high attention weight")
print("Zero dot product → vectors are orthogonal → no attention")
print("Negative dot product → vectors point opposite → softmax pushes weight to ~0")

---

<a id='self-attention'></a>
## 4. Self-Attention

### El salto conceptual más grande

Hasta acá, attention se usaba como un **puente entre encoder y decoder**: el decoder mira los hidden states del encoder. Pero los hidden states que mira todavía fueron construidos secuencialmente por una RNN. Todavía dependemos de la recurrencia.

Ahora viene la pregunta que lo cambió todo:

> **Si attention puede ver todo el input de una vez... ¿para qué necesitamos las RNNs?**

Pensalo. ¿Para qué construimos hidden states secuencialmente con una RNN? Para que cada token tenga "contexto" — información sobre los tokens que vinieron antes. Pero attention ya hace eso: puede mirar cualquier posición del input y construir un resumen ponderado.

### De encoder-decoder attention a self-attention

En encoder-decoder attention:
- **Query**: viene del decoder ("¿qué necesito?")
- **Key/Value**: vienen del encoder ("¿qué hay disponible?")

En **self-attention**, todo viene de la **misma secuencia**:
- **Query**: cada token pregunta "¿qué otros tokens son relevantes para mí?"
- **Key**: cada token responde "esto es lo que yo ofrezco"
- **Value**: cada token contiene "esta es mi información"

### ¿Qué computa self-attention?

Dada una secuencia de tokens $\mathbf{x}_1, \mathbf{x}_2, \ldots, \mathbf{x}_T$, self-attention produce una **nueva representación** de cada token que incorpora información de **todos los demás tokens**.

Para el token $i$:
1. Calcula scores entre $\mathbf{x}_i$ y **todos** los tokens $\mathbf{x}_j$ (incluido él mismo)
2. Softmax → pesos de atención
3. Representación nueva = promedio ponderado de todos los tokens

$$\text{output}_i = \sum_{j=1}^{T} \alpha_{i,j} \cdot \mathbf{x}_j$$

Donde $\alpha_{i,j}$ dice "cuánto debería el token $i$ prestar atención al token $j$".

### Chau secuencialidad

Fijate algo crucial: **no hay recurrencia**. No procesamos un token a la vez. **Todos los tokens se procesan en paralelo**. Cada token mira a todos los demás simultáneamente.

```
RNN (secuencial):          Self-Attention (paralelo):

x1 → h1                   x1  x2  x3  x4
       ↘                    \  |  /  |
x2 → h2                     \ | /   |
       ↘                      ↓↓↓   ↓
x3 → h3                   [todos se miran entre sí]
       ↘                      ↓↓↓   ↓
x4 → h4                   y1  y2  y3  y4

T pasos secuenciales       1 paso paralelo!
```

Esto es una ventaja **enorme** para el entrenamiento. Las RNNs no se pueden paralelizar porque el paso $t$ depende del paso $t-1$. Self-attention calcula todo de una.

### Pero... ¿y el orden?

Si procesamos todos los tokens en paralelo, ¿cómo sabe el modelo que "el" viene antes de "gato"? La respuesta: **no lo sabe, a menos que se lo digamos**.

Self-attention es **invariante al orden** por defecto. Si cambio el orden de los tokens en la entrada, los scores cambian pero la operación es la misma. Para que el modelo sepa el orden, le sumamos **positional embeddings** a cada token:

$$\mathbf{x}_i^{\text{input}} = \mathbf{x}_i^{\text{token embedding}} + \mathbf{p}_i^{\text{positional embedding}}$$

Donde $\mathbf{p}_i$ es un vector que codifica la posición $i$. Puede ser fijo (sinusoidal, como en el Transformer original) o aprendido. El punto es que el orden se inyecta como información adicional, no como estructura de la computación.

Esto es un cambio filosófico enorme respecto a las RNNs: en las RNNs, el orden está en la **estructura de la computación** (procesás de izquierda a derecha). En self-attention, el orden está en los **datos** (positional embeddings). La computación en sí es simétrica.

In [ ]:
# Self-Attention from scratch — simplest version (no Q/K/V projections yet)

def simple_self_attention(X):
    """
    Simplest self-attention: dot product between raw token embeddings.
    
    Args:
        X: (seq_len, dim) — token embeddings
    Returns:
        output: (seq_len, dim) — contextualized representations
        weights: (seq_len, seq_len) — attention matrix
    """
    # Step 1: Compute all pairwise scores
    # X @ X^T: (seq_len, dim) @ (dim, seq_len) = (seq_len, seq_len)
    scores = X @ X.T
    
    # Step 2: Softmax (row-wise) → attention weights
    weights = F.softmax(scores, dim=-1)
    
    # Step 3: Weighted sum → new representations
    # weights @ X: (seq_len, seq_len) @ (seq_len, dim) = (seq_len, dim)
    output = weights @ X
    
    return output, weights


# Demo with a small example
torch.manual_seed(42)

tokens = ["The", "cat", "sat", "on", "the", "mat"]
seq_len = len(tokens)
dim = 8

# Simulated embeddings
X = torch.randn(seq_len, dim)

output, weights = simple_self_attention(X)

print("Simple Self-Attention (no learned projections)")
print("=" * 55)
print(f"\nInput shape:  {X.shape}  ({seq_len} tokens, dim={dim})")
print(f"Output shape: {output.shape}  (same shape — each token now has context!)")
print(f"\nAttention weight matrix ({seq_len}×{seq_len}):")
print(f"  Each row shows how much token i attends to each token j")
print(f"  Each row sums to 1.0")

# Visualize attention matrix
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Attention heatmap
ax = axes[0]
im = ax.imshow(weights.detach().numpy(), cmap='Blues', vmin=0)
ax.set_xticks(range(seq_len))
ax.set_xticklabels(tokens, fontsize=11)
ax.set_yticks(range(seq_len))
ax.set_yticklabels(tokens, fontsize=11)
ax.set_xlabel('Key (attending TO)', fontsize=12)
ax.set_ylabel('Query (attending FROM)', fontsize=12)
ax.set_title('Self-Attention Weights', fontsize=13)

for i in range(seq_len):
    for j in range(seq_len):
        val = weights[i, j].item()
        color = 'white' if val > 0.25 else 'black'
        ax.text(j, i, f'{val:.2f}', ha='center', va='center', fontsize=9, color=color)

plt.colorbar(im, ax=ax)

# Compare input vs output
ax = axes[1]
x_norm = torch.norm(X, dim=-1).detach().numpy()
y_norm = torch.norm(output, dim=-1).detach().numpy()
x_pos = np.arange(seq_len)
width = 0.35
ax.bar(x_pos - width/2, x_norm, width, label='Before attention', color='#3498db', alpha=0.8)
ax.bar(x_pos + width/2, y_norm, width, label='After attention', color='#e74c3c', alpha=0.8)
ax.set_xticks(x_pos)
ax.set_xticklabels(tokens, fontsize=11)
ax.set_ylabel('Vector norm', fontsize=12)
ax.set_title('Token representations before/after self-attention', fontsize=13)
ax.legend(fontsize=11)

plt.tight_layout()
plt.show()

print("\nBefore: each token is an independent embedding (no context).")
print("After: each token's representation is a mixture of ALL tokens, weighted by similarity.")
print("\nProblem: notice the diagonal is dominant — each token attends mostly to itself!")
print("This is why we need learned Q, K, V projections (next section).")

In [ ]:
# Why self-attention attends to itself: the diagonal dominance problem

torch.manual_seed(42)

# Create random embeddings
X = torch.randn(6, 32)

# Self dot products vs cross dot products
scores = X @ X.T

print("Dot product scores (X @ X^T):")
print("\n  Diagonal (self-similarity):")
for i in range(6):
    print(f"    token {i} · token {i} = {scores[i, i].item():.3f}")

print("\n  Off-diagonal (cross-similarity, first row):")
for j in range(6):
    if j != 0:
        print(f"    token 0 · token {j} = {scores[0, j].item():.3f}")

print(f"\n  Average diagonal:     {scores.diag().mean().item():.3f}")
# Create mask for off-diagonal elements
mask = ~torch.eye(6, dtype=torch.bool)
print(f"  Average off-diagonal: {scores[mask].mean().item():.3f}")
print(f"\n  The diagonal is almost always larger!")
print(f"  x · x = ||x||^2 which is always positive,")
print(f"  while x · y can be positive or negative (and averages closer to 0).")
print(f"\n  → After softmax, each token gives the most weight to ITSELF.")
print(f"  → The output is barely different from the input.")
print(f"  → This is why raw self-attention is useless without Q, K, V projections!")

---

<a id='qkv'></a>
## 5. Query, Key, Value con transformaciones aprendibles

### El corazón de la attention moderna

La versión final y definitiva de attention (la que usan los Transformers) agrega una pieza crucial: **transformaciones lineales aprendibles** antes de computar los scores.

En vez de usar los embeddings directamente como queries, keys y values, los **proyectamos** con matrices de pesos:

$$Q = X \cdot W_Q \quad\quad K = X \cdot W_K \quad\quad V = X \cdot W_V$$

Donde:
- $X \in \mathbb{R}^{T \times d_{\text{model}}}$ son los embeddings de entrada (o la salida de la capa anterior)
- $W_Q \in \mathbb{R}^{d_{\text{model}} \times d_k}$ es la matriz de proyección de queries
- $W_K \in \mathbb{R}^{d_{\text{model}} \times d_k}$ es la matriz de proyección de keys
- $W_V \in \mathbb{R}^{d_{\text{model}} \times d_v}$ es la matriz de proyección de values

Cada token se transforma en **tres representaciones distintas** con roles diferentes:

| Representación | Matriz | Rol | Analogía |
|:---------------|:-------|:----|:---------|
| **Query** ($Q$) | $W_Q$ | "¿Qué estoy buscando?" | La pregunta que hago |
| **Key** ($K$) | $W_K$ | "¿Qué tengo para ofrecer?" | La etiqueta que muestro |
| **Value** ($V$) | $W_V$ | "¿Cuál es mi información real?" | El contenido que entrego |

![QKV Attention diagram](../ai_notas/AI%20notas/image%2056.png)

### Scaled Dot-Product Attention — La fórmula completa

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{Q K^T}{\sqrt{d_k}}\right) V$$

Desglosemos cada parte:

**1. $Q K^T$ — Matriz de afinidad**

$$\text{scores} = Q K^T \in \mathbb{R}^{T \times T}$$

Cada elemento $(i, j)$ de esta matriz es el dot product entre el query del token $i$ y el key del token $j$. Mide "¿cuánto debería el token $i$ prestar atención al token $j$?".

**2. $\frac{1}{\sqrt{d_k}}$ — Scaling**

¿Por qué dividir por $\sqrt{d_k}$? Porque el dot product entre vectores de dimensión $d_k$ tiene una varianza que crece con $d_k$. Si $Q$ y $K$ tienen componentes con media 0 y varianza 1:

$$\text{Var}(Q \cdot K) = d_k$$

Si $d_k = 64$, los scores pueden ser del orden de $\pm 8$. Esos valores grandes hacen que softmax se sature (dando pesos casi 0 o casi 1), lo que mata los gradientes. Dividir por $\sqrt{d_k}$ normaliza la varianza a 1.

**3. Softmax — Normalización a distribución de probabilidad**

$$\alpha = \text{softmax}\left(\frac{Q K^T}{\sqrt{d_k}}\right) \in \mathbb{R}^{T \times T}$$

Cada fila es una distribución de probabilidad que suma 1. La fila $i$ contiene los pesos de atención del token $i$ sobre todos los tokens.

**4. $\alpha V$ — Context vectors**

$$\text{output} = \alpha V \in \mathbb{R}^{T \times d_v}$$

Para cada token, el output es un promedio ponderado de los values de todos los tokens, usando los pesos de atención.

### Resumen visual del flujo completo

```
Input X ─┬─ × W_Q ─→ Q ─┐
         │               ├─→ Q @ K^T ──÷ √d_k──→ softmax ──→ weights
         ├─ × W_K ─→ K ─┘                                      │
         │                                                      │
         └─ × W_V ─→ V ──────────────────────────→ weights @ V ─→ Output
```

In [ ]:
# Scaled Dot-Product Attention — FULL step-by-step implementation

torch.manual_seed(42)

# === Setup ===
seq_len = 5
d_model = 16     # embedding dimension
d_k = 8          # query/key dimension
d_v = 8          # value dimension

tokens = ["I", "love", "deep", "learning", "!"]

# Input embeddings (in practice these come from an embedding layer)
X = torch.randn(seq_len, d_model)

# Learned projection matrices
W_Q = torch.randn(d_model, d_k) * 0.1
W_K = torch.randn(d_model, d_k) * 0.1
W_V = torch.randn(d_model, d_v) * 0.1

print("=" * 65)
print("SCALED DOT-PRODUCT ATTENTION — Step by Step")
print("=" * 65)
print(f"\nInput X shape: {X.shape}  ({seq_len} tokens, d_model={d_model})")
print(f"W_Q shape: {W_Q.shape}  (d_model={d_model} → d_k={d_k})")
print(f"W_K shape: {W_K.shape}  (d_model={d_model} → d_k={d_k})")
print(f"W_V shape: {W_V.shape}  (d_model={d_model} → d_v={d_v})")

# === Step 1: Project to Q, K, V ===
print("\n" + "-" * 65)
print("STEP 1: Project X into Q, K, V")
print("-" * 65)

Q = X @ W_Q  # (seq_len, d_k)
K = X @ W_K  # (seq_len, d_k)
V = X @ W_V  # (seq_len, d_v)

print(f"Q = X @ W_Q → shape {Q.shape}")
print(f"K = X @ W_K → shape {K.shape}")
print(f"V = X @ W_V → shape {V.shape}")
print(f"\nSame input X, but 3 different learned projections!")
print(f"Each token now has a query, a key, and a value.")

# === Step 2: Compute attention scores ===
print("\n" + "-" * 65)
print("STEP 2: Compute scores = Q @ K^T")
print("-" * 65)

scores = Q @ K.T  # (seq_len, seq_len)
print(f"scores = Q @ K^T → shape {scores.shape}")
print(f"\nScore matrix (how much each token wants to attend to each other):")
for i, token_i in enumerate(tokens):
    row = [f"{scores[i, j].item():+.2f}" for j in range(seq_len)]
    print(f"  {token_i:>10s}: [{', '.join(row)}]")

# === Step 3: Scale ===
print("\n" + "-" * 65)
print(f"STEP 3: Scale by 1/√d_k = 1/√{d_k} = {1/np.sqrt(d_k):.4f}")
print("-" * 65)

scale = np.sqrt(d_k)
scaled_scores = scores / scale
print(f"Before scaling — score variance: {scores.var().item():.4f}")
print(f"After scaling  — score variance: {scaled_scores.var().item():.4f}")
print(f"\nScaling prevents softmax saturation (gradients stay healthy).")

# === Step 4: Softmax ===
print("\n" + "-" * 65)
print("STEP 4: Softmax → attention weights")
print("-" * 65)

weights = F.softmax(scaled_scores, dim=-1)  # (seq_len, seq_len)
print(f"Attention weights (each row sums to 1):")
for i, token_i in enumerate(tokens):
    row = [f"{weights[i, j].item():.3f}" for j in range(seq_len)]
    total = weights[i].sum().item()
    print(f"  {token_i:>10s}: [{', '.join(row)}]  sum={total:.3f}")

# === Step 5: Weighted sum of values ===
print("\n" + "-" * 65)
print("STEP 5: Output = weights @ V")
print("-" * 65)

output = weights @ V  # (seq_len, d_v)
print(f"output = attention_weights @ V → shape {output.shape}")
print(f"\nEach output token is a weighted combination of ALL value vectors.")
print(f"Token 'love' (row 1) output = {weights[1,0].item():.3f}×V('I') + {weights[1,1].item():.3f}×V('love') + ...")
print(f"\nThis IS the contextualized representation!")

In [ ]:
# Visualize the full attention pipeline

fig = plt.figure(figsize=(16, 10))
gs = GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.3)

# 1. Q matrix
ax = fig.add_subplot(gs[0, 0])
im = ax.imshow(Q.detach().numpy(), cmap='RdBu_r', aspect='auto')
ax.set_title('Q = X @ W_Q\n(what each token asks)', fontsize=11)
ax.set_yticks(range(seq_len))
ax.set_yticklabels(tokens)
ax.set_xlabel(f'd_k = {d_k}')
plt.colorbar(im, ax=ax)

# 2. K matrix
ax = fig.add_subplot(gs[0, 1])
im = ax.imshow(K.detach().numpy(), cmap='RdBu_r', aspect='auto')
ax.set_title('K = X @ W_K\n(what each token offers)', fontsize=11)
ax.set_yticks(range(seq_len))
ax.set_yticklabels(tokens)
ax.set_xlabel(f'd_k = {d_k}')
plt.colorbar(im, ax=ax)

# 3. V matrix
ax = fig.add_subplot(gs[0, 2])
im = ax.imshow(V.detach().numpy(), cmap='RdBu_r', aspect='auto')
ax.set_title('V = X @ W_V\n(real info each token holds)', fontsize=11)
ax.set_yticks(range(seq_len))
ax.set_yticklabels(tokens)
ax.set_xlabel(f'd_v = {d_v}')
plt.colorbar(im, ax=ax)

# 4. Attention scores (before softmax)
ax = fig.add_subplot(gs[1, 0])
im = ax.imshow(scaled_scores.detach().numpy(), cmap='RdBu_r', aspect='auto')
ax.set_title('QK^T / √d_k\n(affinity scores)', fontsize=11)
ax.set_yticks(range(seq_len))
ax.set_yticklabels(tokens)
ax.set_xticks(range(seq_len))
ax.set_xticklabels(tokens, rotation=45)
plt.colorbar(im, ax=ax)

# 5. Attention weights (after softmax)
ax = fig.add_subplot(gs[1, 1])
im = ax.imshow(weights.detach().numpy(), cmap='Blues', vmin=0, aspect='auto')
ax.set_title('softmax(QK^T / √d_k)\n(attention weights)', fontsize=11)
ax.set_yticks(range(seq_len))
ax.set_yticklabels(tokens)
ax.set_xticks(range(seq_len))
ax.set_xticklabels(tokens, rotation=45)
for i in range(seq_len):
    for j in range(seq_len):
        val = weights[i, j].item()
        color = 'white' if val > 0.3 else 'black'
        ax.text(j, i, f'{val:.2f}', ha='center', va='center', fontsize=9, color=color)
plt.colorbar(im, ax=ax)

# 6. Output
ax = fig.add_subplot(gs[1, 2])
im = ax.imshow(output.detach().numpy(), cmap='RdBu_r', aspect='auto')
ax.set_title('Output = weights @ V\n(contextualized tokens)', fontsize=11)
ax.set_yticks(range(seq_len))
ax.set_yticklabels(tokens)
ax.set_xlabel(f'd_v = {d_v}')
plt.colorbar(im, ax=ax)

plt.suptitle('Scaled Dot-Product Attention — Complete Pipeline', fontsize=15, y=1.02)
plt.show()

In [ ]:
# Clean PyTorch implementation of Scaled Dot-Product Attention

class ScaledDotProductAttention(nn.Module):
    """Attention(Q, K, V) = softmax(QK^T / √d_k) V"""
    
    def __init__(self, d_model, d_k, d_v):
        super().__init__()
        self.W_Q = nn.Linear(d_model, d_k, bias=False)
        self.W_K = nn.Linear(d_model, d_k, bias=False)
        self.W_V = nn.Linear(d_model, d_v, bias=False)
        self.scale = d_k ** 0.5
    
    def forward(self, X, mask=None):
        """
        Args:
            X: (batch, seq_len, d_model) — input embeddings
            mask: optional (batch, seq_len, seq_len) — attention mask
        Returns:
            output: (batch, seq_len, d_v)
            weights: (batch, seq_len, seq_len)
        """
        Q = self.W_Q(X)  # (batch, seq_len, d_k)
        K = self.W_K(X)  # (batch, seq_len, d_k)
        V = self.W_V(X)  # (batch, seq_len, d_v)
        
        # Scores: Q @ K^T / √d_k
        scores = torch.bmm(Q, K.transpose(1, 2)) / self.scale  # (batch, seq_len, seq_len)
        
        # Optional mask (for causal/padding)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        
        # Softmax → weights
        weights = F.softmax(scores, dim=-1)  # (batch, seq_len, seq_len)
        
        # Weighted sum of values
        output = torch.bmm(weights, V)  # (batch, seq_len, d_v)
        
        return output, weights


# Test it
torch.manual_seed(42)

batch_size = 2
seq_len = 6
d_model = 32
d_k = d_v = 16

attn = ScaledDotProductAttention(d_model, d_k, d_v)
X = torch.randn(batch_size, seq_len, d_model)

output, weights = attn(X)

print("Scaled Dot-Product Attention Module")
print(f"  Input:   {X.shape}  (batch={batch_size}, seq_len={seq_len}, d_model={d_model})")
print(f"  Output:  {output.shape}  (batch={batch_size}, seq_len={seq_len}, d_v={d_v})")
print(f"  Weights: {weights.shape}  (batch={batch_size}, seq_len={seq_len}, seq_len={seq_len})")
print(f"  Params:  {sum(p.numel() for p in attn.parameters())}")
print(f"\n  Each token's output is a weighted mix of all tokens' values.")
print(f"  The weights are determined by query-key similarity.")

# Demo with causal mask (for autoregressive models like GPT)
print("\n--- Causal Mask (for autoregressive decoding) ---")
causal_mask = torch.tril(torch.ones(seq_len, seq_len)).unsqueeze(0)  # (1, T, T)
output_masked, weights_masked = attn(X, mask=causal_mask)

print(f"Causal mask (token can only attend to previous tokens):")
print(causal_mask[0].numpy().astype(int))
print(f"\nMasked attention weights (batch 0):")
print(weights_masked[0].detach().numpy().round(3))
print(f"\nNotice: upper triangle is 0 — no attending to future tokens!")

### Scaling: ¿por qué dividir por $\sqrt{d_k}$?

Vamos a ver empíricamente por qué es necesario el scaling. Sin él, los scores se hacen muy grandes cuando $d_k$ crece, y softmax se satura.

In [ ]:
# Why scaling matters: empirical demo

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

for ax, d_k_test in zip(axes, [4, 64, 512]):
    torch.manual_seed(42)
    q = torch.randn(1, 10, d_k_test)  # 10 queries
    k = torch.randn(1, 10, d_k_test)  # 10 keys
    
    # Without scaling
    scores_raw = torch.bmm(q, k.transpose(1, 2))[0]  # (10, 10)
    weights_raw = F.softmax(scores_raw, dim=-1)
    
    # With scaling
    scores_scaled = scores_raw / np.sqrt(d_k_test)
    weights_scaled = F.softmax(scores_scaled, dim=-1)
    
    # Plot distributions of weights
    w_raw = weights_raw.flatten().detach().numpy()
    w_scaled = weights_scaled.flatten().detach().numpy()
    
    ax.hist(w_raw, bins=30, alpha=0.6, label=f'No scaling', color='#e74c3c', density=True)
    ax.hist(w_scaled, bins=30, alpha=0.6, label=f'With √d_k', color='#3498db', density=True)
    ax.set_title(f'd_k = {d_k_test}\nscore var: {scores_raw.var().item():.1f} → {scores_scaled.var().item():.1f}', fontsize=11)
    ax.set_xlabel('Attention weight', fontsize=10)
    ax.legend(fontsize=9)
    ax.set_xlim(-0.05, 1.0)

plt.suptitle('Effect of Scaling on Attention Weight Distribution', fontsize=13, y=1.05)
plt.tight_layout()
plt.show()

print("Without scaling, as d_k grows:")
print("  - Scores get larger → softmax saturates → weights become 0-or-1")
print("  - This means attention becomes 'hard' (looks at one token only)")
print("  - Gradients vanish through softmax → training breaks")
print("\nWith √d_k scaling:")
print("  - Score variance stays ~1 regardless of d_k")
print("  - Softmax produces 'soft' weights → smooth gradients → stable training")

---

<a id='por-que-qkv'></a>
## 6. ¿Por qué Q, K, V distintos?

### La pregunta natural

Podrías preguntarte: ¿por qué tres matrices separadas? ¿No alcanza con usar los embeddings directamente? Ya vimos que la self-attention sin proyecciones tiene un problema (la diagonal domina), pero profundicemos.

### Problema: sin transformación, el dot product es aburrido

Si usamos los embeddings directamente (sin $W_Q, W_K, W_V$):

$$\text{score}(x_i, x_j) = x_i^T \cdot x_j$$

El dot product de un vector consigo mismo es $\|x_i\|^2$, que siempre es positivo y generalmente el más grande. Después del softmax, cada token le presta más atención a sí mismo que a cualquier otro. **La representación de salida es casi igual a la de entrada.** No aprendimos nada.

### Solución: Q, K, V dan roles distintos

Al separar en tres proyecciones, le damos al modelo la capacidad de **especializar cada rol**:

**$W_Q$ — Optimizado para preguntar**

La query de un token codifica "¿qué tipo de información necesito?" Si estoy en la palabra "comió", mi query puede codificar "necesito saber QUIÉN comió" y "QUÉ comió". $W_Q$ aprende a transformar el embedding en una representación que capture estas necesidades.

**$W_K$ — Optimizado para ser encontrable**

El key de un token codifica "¿qué tipo de información ofrezco?" Si soy la palabra "gato", mi key puede codificar "soy un sustantivo, soy el sujeto". $W_K$ aprende a transformar el embedding en una representación que facilite ser encontrado por las queries relevantes.

**$W_V$ — Optimizado para informar**

El value de un token codifica "¿cuál es mi información real?" Si soy la palabra "gato", mi value contiene la información semántica completa que necesita el token que me está buscando. $W_V$ aprende a transformar el embedding en la representación más útil para transmitir información.

### Analogía concreta: sistema de búsqueda

Pensalo como buscar en Google:

| Componente | En Google | En Attention |
|:-----------|:----------|:-------------|
| **Query** | Lo que escribís en la barra de búsqueda | Lo que un token necesita saber |
| **Key** | El título/metadata de cada página | Lo que cada token ofrece al matching |
| **Value** | El contenido real de la página | La información real que se extrae |

Cuando buscás "receta tiramisú", Google compara tu **query** contra los **keys** (títulos/metadata) de millones de páginas. Las que mejor matchean obtienen el peso más alto. Pero lo que te muestra es el **value** (el contenido de la página), no el key.

Si query y key fueran lo mismo, sería como si Google te mostrara la barra de búsqueda en vez del contenido de la página. No sirve.

In [ ]:
# Empirical proof: Q, K, V projections break diagonal dominance

torch.manual_seed(42)

seq_len = 8
d_model = 32
d_k = 16

X = torch.randn(seq_len, d_model)

# === Without projections ===
scores_raw = X @ X.T
weights_raw = F.softmax(scores_raw / np.sqrt(d_model), dim=-1)

# === With learned Q, K projections ===
W_Q = torch.randn(d_model, d_k) * 0.3
W_K = torch.randn(d_model, d_k) * 0.3

Q = X @ W_Q
K = X @ W_K
scores_proj = Q @ K.T
weights_proj = F.softmax(scores_proj / np.sqrt(d_k), dim=-1)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Without projections
ax = axes[0]
im = ax.imshow(weights_raw.detach().numpy(), cmap='Blues', vmin=0, vmax=0.5)
ax.set_title('Without Q,K projections\n(X @ X^T)', fontsize=12)
ax.set_xlabel('Key')
ax.set_ylabel('Query')
for i in range(seq_len):
    for j in range(seq_len):
        val = weights_raw[i, j].item()
        ax.text(j, i, f'{val:.2f}', ha='center', va='center', fontsize=8,
                color='white' if val > 0.25 else 'black')
plt.colorbar(im, ax=ax)

# With projections  
ax = axes[1]
im = ax.imshow(weights_proj.detach().numpy(), cmap='Blues', vmin=0, vmax=0.5)
ax.set_title('With Q,K projections\n(XW_Q @ (XW_K)^T)', fontsize=12)
ax.set_xlabel('Key')
ax.set_ylabel('Query')
for i in range(seq_len):
    for j in range(seq_len):
        val = weights_proj[i, j].item()
        ax.text(j, i, f'{val:.2f}', ha='center', va='center', fontsize=8,
                color='white' if val > 0.25 else 'black')
plt.colorbar(im, ax=ax)

# Diagonal dominance comparison
ax = axes[2]
diag_raw = weights_raw.diag().detach().numpy()
diag_proj = weights_proj.diag().detach().numpy()
x = np.arange(seq_len)
ax.bar(x - 0.2, diag_raw, 0.35, label='No projection', color='#e74c3c', alpha=0.8)
ax.bar(x + 0.2, diag_proj, 0.35, label='With Q,K projection', color='#3498db', alpha=0.8)
ax.axhline(y=1/seq_len, color='gray', linestyle='--', label=f'Uniform = {1/seq_len:.2f}')
ax.set_xlabel('Token', fontsize=11)
ax.set_ylabel('Self-attention weight', fontsize=11)
ax.set_title('Diagonal dominance\n(weight each token gives to itself)', fontsize=12)
ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

print(f"Without projections:")
print(f"  Average self-attention (diagonal): {diag_raw.mean():.3f}")
print(f"  Uniform would be:                  {1/seq_len:.3f}")
print(f"  → Each token attends mostly to ITSELF")
print(f"\nWith Q,K projections:")
print(f"  Average self-attention (diagonal): {diag_proj.mean():.3f}")
print(f"  → Attention is distributed more evenly → tokens actually mix information!")

In [ ]:
# Why separate V matters: the information you extract should differ from
# the information used for matching

torch.manual_seed(42)

# Simple example: imagine tokens have two aspects — "role" and "content"
# The key should encode the role (for matching), the value should encode the content

seq_len = 4
d_model = 8
d_k = 4
d_v = 4

# Simulated embeddings
X = torch.randn(seq_len, d_model)

# Case 1: K == V (no separate value projection)
W_Q_1 = torch.randn(d_model, d_k) * 0.2
W_K_1 = torch.randn(d_model, d_k) * 0.2

Q_1 = X @ W_Q_1
K_1 = X @ W_K_1
V_1 = K_1  # V == K ← same as Luong

scores_1 = Q_1 @ K_1.T / np.sqrt(d_k)
weights_1 = F.softmax(scores_1, dim=-1)
output_1 = weights_1 @ V_1

# Case 2: Separate V projection
W_V_2 = torch.randn(d_model, d_v) * 0.2  # separate!
V_2 = X @ W_V_2

output_2 = weights_1 @ V_2  # same weights, different values

print("Case 1 (K == V):")
print(f"  What you match against and what you extract are THE SAME.")
print(f"  Output cosine similarity to keys: {F.cosine_similarity(output_1.flatten().unsqueeze(0), K_1.flatten().unsqueeze(0)).item():.3f}")

print(f"\nCase 2 (separate V):")
print(f"  Matching (Q@K^T) determines WHERE to look.")
print(f"  V determines WHAT to extract.")
print(f"  Output cosine similarity to keys: {F.cosine_similarity(output_2.flatten().unsqueeze(0), K_1.flatten().unsqueeze(0)).item():.3f}")

print(f"\nWith separate V, the model can:")
print(f"  - Use K to encode 'I am a noun/verb/adjective' (for matching)")
print(f"  - Use V to encode the actual semantic content (for extraction)")
print(f"  - These are fundamentally different types of information!")

### Resumen: la evolución de los roles

| Versión | Query | Key | Value | Problema |
|:--------|:------|:----|:------|:---------|
| **Raw self-attention** | $x_i$ | $x_j$ | $x_j$ | Diagonal domina, no aprende nada |
| **Luong (dot product)** | $s_{t-1}$ (decoder) | $h_j$ (encoder) | $h_j$ (encoder) | K == V, matching y extracción mezclados |
| **QKV Attention** | $x_i W_Q$ | $x_j W_K$ | $x_j W_V$ | Cada rol especializado, máxima expresividad |

Las matrices $W_Q, W_K, W_V$ le dan al modelo la capacidad de **decidir qué información usar para buscar vs. qué información extraer**. Esta separación es fundamental para que attention funcione bien.

---

<a id='capas'></a>
## 7. Múltiples capas de attention

### ¿Una sola capa de attention es suficiente?

No. Una sola capa de self-attention computa un promedio ponderado de los embeddings originales. Puede capturar relaciones directas entre tokens ("el" → "gato", "gato" → "negro"), pero no puede capturar relaciones que requieren **razonamiento en cadena**.

Pensá en esta oración:

> "El gato que persiguió al ratón que robó el queso estaba cansado"

Para entender que "cansado" se refiere a "gato" (y no a "ratón" o "queso"), necesitás:
1. Primero conectar "estaba" con su sujeto a través de la cláusula subordinada
2. Eso requiere entender la estructura de las cláusulas
3. Que a su vez requiere conectar "que" con sus referentes

Esto es razonamiento multi-paso. Una capa de attention no puede hacerlo.

### Capa por capa: representaciones progresivas

Al apilar capas de attention, cada capa opera sobre la **salida de la capa anterior**, no sobre los embeddings originales:

```
Capa 0 (embeddings):      x1      x2      x3      x4      x5
                           │       │       │       │       │
                           ▼       ▼       ▼       ▼       ▼
                          ┌─────────────────────────────────┐
Capa 1 (attention):       │      self-attention + FFN       │
                          └─────────────────────────────────┘
                           │       │       │       │       │
                           ▼       ▼       ▼       ▼       ▼
                          z1(1)   z2(1)   z3(1)   z4(1)   z5(1)
                           │       │       │       │       │
                           ▼       ▼       ▼       ▼       ▼
                          ┌─────────────────────────────────┐
Capa 2 (attention):       │      self-attention + FFN       │
                          └─────────────────────────────────┘
                           │       │       │       │       │
                           ▼       ▼       ▼       ▼       ▼
                          z1(2)   z2(2)   z3(2)   z4(2)   z5(2)
```

### Qué hace cada capa

| Capa | Input | Qué captura | Analogía |
|:-----|:------|:------------|:---------|
| **0** | Embeddings crudos | Cada token es independiente | Palabras sueltas |
| **1** | Embeddings originales | Relaciones directas entre tokens vecinos y relevantes | "El gato" se conecta, "negro" se une a "gato" |
| **2** | Representaciones ya mezcladas de capa 1 | Relaciones entre grupos — frases, cláusulas | "El gato negro" como unidad se conecta con "duerme" |
| **3+** | Representaciones aún más mezcladas | Relaciones de alto nivel — tema, intención, semántica global | Entender la oración completa y sus implicaciones |

### La clave: cada capa tiene contexto de las anteriores

En la capa 1, cuando el token "cansado" hace attention sobre "gato", obtiene la representación raw de "gato". Pero en la capa 2, cuando "cansado" hace attention sobre "gato", obtiene la representación de "gato" que ya incluye información de "persiguió", "ratón", etc. (de la capa 1).

Es como una cadena de razonamiento:
- **Capa 1**: "gato" ← se enriquece con "el", "que", "persiguió"
- **Capa 2**: "cansado" ← se conecta con el "gato" ya enriquecido
- Resultado: "cansado" ahora sabe que se refiere al gato que persiguió al ratón

### Recurrencia implícita, sin vanishing gradients

Esto resuelve un problema fundamental. Las RNNs necesitaban recurrencia (procesar secuencialmente) para construir representaciones contextuales. Pero la recurrencia causaba vanishing gradients.

Las capas de attention logran lo mismo — representaciones progresivamente más contextuales — pero con dos ventajas:

1. **Full paralelización**: no hay dependencia temporal. Todos los tokens se procesan en paralelo en cada capa.

2. **Sin vanishing gradients por profundidad**: gracias a las residual connections (que veremos en detalle en el notebook de Transformers), el gradiente puede fluir directamente de la última capa a la primera sin degradarse.

Es como si tuvieras una RNN de $L$ pasos (donde $L$ es el número de capas), pero sin los problemas de las RNNs. La "recurrencia" está en la profundidad, no en el tiempo.

In [ ]:
# Demo: stacking attention layers and seeing how representations evolve

class AttentionLayer(nn.Module):
    """One layer of self-attention + feedforward."""
    def __init__(self, d_model, d_k, d_ff=None):
        super().__init__()
        if d_ff is None:
            d_ff = d_model * 4
        
        self.W_Q = nn.Linear(d_model, d_k, bias=False)
        self.W_K = nn.Linear(d_model, d_k, bias=False)
        self.W_V = nn.Linear(d_model, d_model, bias=False)  # d_v = d_model for residual
        self.scale = d_k ** 0.5
        
        # Feed-forward network
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Linear(d_ff, d_model)
        )
        
        # Layer norms
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
    
    def forward(self, X):
        # Self-attention with residual connection
        Q = self.W_Q(X)
        K = self.W_K(X)
        V = self.W_V(X)
        
        scores = torch.bmm(Q, K.transpose(1, 2)) / self.scale
        weights = F.softmax(scores, dim=-1)
        attn_output = torch.bmm(weights, V)
        
        X = self.norm1(X + attn_output)  # residual + norm
        
        # FFN with residual connection
        X = self.norm2(X + self.ffn(X))  # residual + norm
        
        return X, weights


# Stack multiple layers
torch.manual_seed(42)

num_layers = 4
d_model = 32
d_k = 16
seq_len = 8
batch_size = 1

layers = nn.ModuleList([AttentionLayer(d_model, d_k) for _ in range(num_layers)])

# Input
tokens = ["The", "cat", "that", "chased", "the", "mouse", "was", "tired"]
X = torch.randn(batch_size, seq_len, d_model)

# Forward through all layers, collecting attention patterns
layer_weights = []
layer_representations = [X.detach().clone()]

current = X
for i, layer in enumerate(layers):
    current, weights = layer(current)
    layer_weights.append(weights.detach())
    layer_representations.append(current.detach().clone())

# Visualize attention patterns at each layer
fig, axes = plt.subplots(1, num_layers, figsize=(20, 5))

for i, (ax, w) in enumerate(zip(axes, layer_weights)):
    im = ax.imshow(w[0].numpy(), cmap='Blues', vmin=0, aspect='auto')
    ax.set_title(f'Layer {i+1}', fontsize=13)
    ax.set_xticks(range(seq_len))
    ax.set_xticklabels(tokens, rotation=45, ha='right', fontsize=9)
    ax.set_yticks(range(seq_len))
    ax.set_yticklabels(tokens, fontsize=9)
    if i == 0:
        ax.set_ylabel('Query', fontsize=11)
    ax.set_xlabel('Key', fontsize=11)

plt.suptitle('Attention Patterns Across Layers\n("The cat that chased the mouse was tired")', fontsize=14, y=1.05)
plt.tight_layout()
plt.show()

print("Each layer shows a different attention pattern.")
print("Early layers: local, syntactic connections.")
print("Later layers: broader, more semantic connections.")

In [ ]:
# How similar are token representations at each layer?
# In early layers, tokens are independent. In later layers, they mix.

fig, axes = plt.subplots(1, num_layers + 1, figsize=(22, 4))

for i, (ax, rep) in enumerate(zip(axes, layer_representations)):
    # Cosine similarity matrix between all tokens
    rep_normalized = F.normalize(rep[0], dim=-1)  # (seq_len, d_model)
    sim = (rep_normalized @ rep_normalized.T).numpy()
    
    im = ax.imshow(sim, cmap='RdYlBu_r', vmin=-1, vmax=1, aspect='auto')
    layer_name = f'Layer {i}' if i > 0 else 'Embeddings\n(layer 0)'
    ax.set_title(layer_name, fontsize=11)
    ax.set_xticks(range(seq_len))
    ax.set_xticklabels(tokens, rotation=45, ha='right', fontsize=8)
    ax.set_yticks(range(seq_len))
    ax.set_yticklabels(tokens, fontsize=8)

plt.suptitle('Token Similarity Across Layers (cosine similarity)', fontsize=14, y=1.05)
plt.tight_layout()
plt.show()

# Measure average off-diagonal similarity at each layer
print("Average off-diagonal cosine similarity (how much tokens mix):")
for i, rep in enumerate(layer_representations):
    rep_normalized = F.normalize(rep[0], dim=-1)
    sim = (rep_normalized @ rep_normalized.T)
    mask = ~torch.eye(seq_len, dtype=torch.bool)
    avg_sim = sim[mask].mean().item()
    layer_name = f'Embeddings (layer 0)' if i == 0 else f'Layer {i}'
    bar = '█' * int(abs(avg_sim) * 50)
    print(f"  {layer_name:22s}: {avg_sim:+.4f} {bar}")

print("\nAs we go deeper, tokens become more similar to each other.")
print("This is because each layer mixes information between tokens.")
print("By the last layer, each token 'knows about' the entire sequence.")

In [ ]:
# Effective receptive field: how far can information travel in L layers?
# In an RNN: L steps of recurrence → info can travel L positions.
# In attention: info can travel ANYWHERE in 1 layer, but multi-hop reasoning needs multiple layers.

print("Comparison: RNN vs Self-Attention")
print("=" * 60)
print()
print("For a sequence of T tokens:")
print()
print("  RNN:")
print("    - Processing: O(T) sequential steps")
print("    - Max path between any two tokens: O(T)")
print("    - Each step: O(H²) computation")
print("    - Total: O(T × H²)")
print("    - Parallelizable: NO ✗")
print()
print("  Self-Attention:")
print("    - Processing: O(1) parallel step")
print("    - Max path between any two tokens: O(1) ← any token can attend to any other!")
print("    - Each step: O(T² × d) computation (all pairs)")
print("    - Total: O(T² × d)")
print("    - Parallelizable: YES ✓")
print()
print("  Trade-off:")
print("    Self-attention is O(T²) in compute — quadratic in sequence length.")
print("    This is more expensive than RNN for very long sequences.")
print("    But it's MUCH more parallelizable → faster on GPUs.")
print("    And the O(1) path length means better gradient flow.")
print()

# Illustrate with concrete numbers
print("\nConcrete example (H=d=512):")
print(f"{'T':>6s} | {'RNN (T×H²)':>15s} | {'Attention (T²×d)':>18s} | {'Winner':>10s}")
print("-" * 60)
H = 512
for T in [10, 100, 512, 1000, 4096]:
    rnn_cost = T * H * H
    attn_cost = T * T * H
    winner = "RNN" if rnn_cost < attn_cost else "Attention" if attn_cost < rnn_cost else "Tie"
    print(f"{T:>6d} | {rnn_cost:>15,d} | {attn_cost:>18,d} | {winner:>10s}")

print(f"\nAttention wins when T < H (common: typical T=512, H=512).")
print(f"RNN wins when T >> H (very long sequences).")
print(f"But attention is still preferred because of parallelization + gradient flow.")

---

<a id='conclusiones'></a>
## 8. Conclusiones

### La evolución: de RNN hidden states a QKV attention

Recorrimos la historia de una de las ideas más importantes del deep learning. Lo que empezó como un parche para mejorar la traducción automática terminó siendo la base de **toda** la inteligencia artificial moderna.

### Resumen de los conceptos clave

| Concepto | Idea central | Limitación |
|:---------|:-------------|:-----------|
| **Hidden state bottleneck** | Todo comprimido en un vector → se pierde info | Motivó la creación de attention |
| **Bahdanau attention** | Red feedforward que aprende scores de relevancia | Funciona pero es lento (red extra) |
| **Dot product attention (Luong)** | Score = dot product. Simple, rápido | K == V, no separa matching de extracción |
| **Self-attention** | Cada token mira a todos los otros en la misma secuencia. Chau RNNs | Sin posición inherente, necesita positional embeddings |
| **QKV attention** | Tres proyecciones aprendibles: Q busca, K ofrece, V informa | Base de los Transformers |
| **Múltiples capas** | Capas apiladas refinan progresivamente el contexto | Más capas = más cómputo |

### Las victorias clave de attention sobre RNNs

| Aspecto | RNN / LSTM | Self-Attention |
|:--------|:-----------|:---------------|
| **Paralelización** | No (secuencial) | Sí (todo en paralelo) |
| **Path length** | O(T) para conectar tokens distantes | O(1) — conexión directa |
| **Vanishing gradients** | Sí (incluso con LSTM, gradual) | No (residual connections) |
| **Memoria** | Comprimida en un vector fijo | Acceso directo a toda la secuencia |
| **Complejidad** | O(T × H²) | O(T² × d) — cuadrático en T |
| **En la práctica** | Más eficiente para secuencias muy largas | Más rápido en GPU, mejor calidad |

### ¿Qué viene después?

Attention es el ingrediente principal, pero hay varias piezas más que necesitamos para armar el Transformer completo:

- **Multi-Head Attention**: en vez de una sola operación de attention, ejecutar varias en paralelo con distintas proyecciones (distintos "ojos" mirando distintos aspectos)
- **Positional Encoding**: cómo inyectar información de posición
- **Layer Normalization**: normalización que estabiliza el entrenamiento
- **Residual Connections**: atajos que permiten gradientes directos
- **Feed-Forward Networks**: procesamiento posición por posición entre capas de attention

Todo esto se combina en la arquitectura **Transformer**, que veremos en el próximo notebook.

---

**Siguiente notebook →** [11 - Transformers](./11_transformers.ipynb): la arquitectura que unifica todas estas ideas en un diseño elegante que conquistó el mundo del deep learning.